In [1]:
print("MCDIA500 • Fase 2: Exploración, Preprocesamiento y Validación • Grupo 5")

MCDIA500 • Fase 2: Exploración, Preprocesamiento y Validación • Grupo 5


# F2 — Exploración, preprocesamiento, transformación y validación del dataset
## Ofertas en licitaciones públicas del sector Salud de marzo de 2026
**Curso:** MCDIA500, Programación para la Ciencia de Datos. **Grupo:** 5.  
**Integrantes:** Víctor Bravo Barrera, Nayadeth Garrido y Alexander Sepulveda.

Esta fase materializa el pipeline de ingeniería y preparación de datos del proyecto, cumpliendo con los estándares de reproducibilidad, modularidad y rigor analítico definidos en la Fase 1:
- **Obtención y trazabilidad:** Carga verificada mediante hash SHA-256 del dataset oficial de ChileCompra.
- **Exploración inicial (EDA):** Diagnóstico de completitud, tipos de datos, cardinalidad y valores faltantes.
- **Limpieza rigurosa:** Exclusión justificada de columnas 100% vacías y neutralización de fechas anómalas (año 1900).
- **Transformación de tipos:** Casting de fechas a `datetime64[ns]` y normalización de variables categóricas.
- **Variables derivadas:** Generación de indicadores analíticos (`oferta_ganadora`, `licitacion_adjudicada`, `plazo_cierre_dias`).
- **Validación técnica:** Suite de pruebas automatizadas (casos normales, casos límite y excepciones controladas).
- **Exportación reproducible:** Almacenamiento en `data/processed/` con trazabilidad criptográfica.

## 1. Configuración del entorno y funciones modulares
Se carga el entorno de trabajo y se importan las funciones implementadas en `src/proyecto.py`, garantizando la separación entre lógica de procesamiento y presentación interactiva.

In [2]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

raiz = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "proyecto.py").is_file() and (p / "F2").is_dir()), None)
if raiz is None:
    raise FileNotFoundError("Abrir el notebook desde la raíz del repositorio o F2.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.proyecto import (
    versiones_entorno, sha256_archivo, leer_datos_f1,
    resumen_exploracion, limpiar_datos_f2, validar_dataset_procesado,
    exportar_datos_procesados
)

print("Directorio raíz del proyecto:", raiz)
display(versiones_entorno())

Directorio raíz del proyecto: C:\Trabajos\sumativo-1


{'numpy': '2.3.5',
 'pandas': '3.0.1',
 'jupyterlab': '4.6.3',
 'ipykernel': '7.3.0',
 'nbformat': '5.11.1',
 'nbconvert': '7.17.1',
 'nbclient': '0.11.0'}

## 2. Obtención y verificación criptográfica de los datos de entrada
Se verifica la huella digital SHA-256 del dataset original `licitaciones_salud_marzo_2026.csv` antes de procesar para asegurar que el insumo no haya sido alterado.

In [3]:
ruta_raw = raiz / "data" / "raw" / "licitaciones_salud_marzo_2026.csv"
huella_esperada = "490d9209a10d387011d481b72b7891f26e997974ec2cf9dfc518aa4a08552232"
huella_actual = sha256_archivo(ruta_raw)
assert huella_actual == huella_esperada, "Error: la huella SHA-256 del CSV no coincide con la versión documentada."

columnas_esquema = ["NroLicitacion", "TipoLicitacion", "TamanoProveedor", "ResultadoOferta", "EstadoLicitacion", "Sector", "FechaPublicacion"]
df_raw = leer_datos_f1(ruta_raw, columnas_esquema)
print("Dataset de entrada cargado exitosamente:")
print("- Filas totales:", len(df_raw))
print("- Columnas totales:", len(df_raw.columns))
print("- SHA-256 verificado:", huella_actual)
assert df_raw.shape == (44226, 74)

Dataset de entrada cargado exitosamente:
- Filas totales: 44226
- Columnas totales: 74
- SHA-256 verificado: 490d9209a10d387011d481b72b7891f26e997974ec2cf9dfc518aa4a08552232


## 3. Exploración inicial (EDA) y diagnóstico de calidad
Mediante `resumen_exploracion`, analizamos la completitud del conjunto de datos, identificando columnas sin variabilidad o completamente nulas.

In [4]:
metricas_eda = resumen_exploracion(df_raw)
df_metricas = pd.DataFrame([
    {"Métrica": "Total de filas observadas", "Valor": metricas_eda["total_filas"]},
    {"Métrica": "Total de columnas", "Valor": metricas_eda["total_columnas"]},
    {"Métrica": "Columnas completas (sin nulos)", "Valor": metricas_eda["columnas_completas"]},
    {"Métrica": "Columnas con valores nulos parciales", "Valor": metricas_eda["columnas_con_nulos"]},
    {"Métrica": "Columnas 100% vacías", "Valor": len(metricas_eda["columnas_100_nulos"])},
    {"Métrica": "Columnas con valor único (constantes)", "Valor": len(metricas_eda["columnas_constantes"])},
])
display(df_metricas)
print("Columnas 100% vacías identificadas para exclusión:", metricas_eda["columnas_100_nulos"])
print("Columnas con valor único:", metricas_eda["columnas_constantes"])
print("Duplicados exactos:", int(df_raw.duplicated().sum()))
print("Licitaciones distintas:", df_raw.NroLicitacion.nunique())


,Métrica,Valor
0,Total de filas observadas,44226
1,Total de columnas,74
2,Columnas completas (sin nulos),49
3,Columnas con valores nulos parciales,22
4,Columnas 100% vacías,3
5,Columnas con valor único (constantes),7


Columnas 100% vacías identificadas para exclusión: ['LicitacionBaseTipo', 'ContratoRenovable', 'UnidadTiempoRenovacion']
Columnas con valor único: ['LicitacionInformada', 'LicitacionBaseTipo', 'TipoAdjudicacion', 'ContratoRenovable', 'ValorTiempoRenovacion', 'UnidadTiempoRenovacion', 'Sector']


Duplicados exactos: 0


Licitaciones distintas: 1864


### Distribuciones de las variables clave del estudio
Examinamos la distribución de frecuencias de las variables principales (`TipoLicitacion`, `TamanoProveedor`, `ResultadoOferta`) y su interacción con `EstadoLicitacion`.

In [5]:
from src.proyecto import tablas_frecuencia

for columna, tabla in tablas_frecuencia(df_raw, ["TipoLicitacion", "TamanoProveedor", "ResultadoOferta"]).items():
    print("Distribución de", columna)
    display(tabla)
print("EstadoLicitacion frente a ResultadoOferta:")
display(pd.crosstab(df_raw["EstadoLicitacion"], df_raw["ResultadoOferta"], margins=True))

Distribución de TipoLicitacion


,Frecuencia
TipoLicitacion,
Licitación Pública Entre 100 y 1000 UTM (LE),23708
Licitación Pública Mayor 1000 UTM (LP),14448
Licitación Pública Mayor a 5000 (LR),3227
Licitación Pública Menor a 100 UTM (L1),2757
Licitación Privada Mayor a 1000 UTM,72
Licitación Privada entre 100 y 1000 UTM.,12
Licitación Privada Mayor a 5000 (I2),2


Distribución de TamanoProveedor


,Frecuencia
TamanoProveedor,
Grande,18865
Pequeña,9682
Mediana,9015
NoClasificado,3410
Micro,3254


Distribución de ResultadoOferta


,Frecuencia
ResultadoOferta,
Ganadora,26063
Perdedora,18163


EstadoLicitacion frente a ResultadoOferta:


ResultadoOferta,Ganadora,Perdedora,All
EstadoLicitacion,,,
Adjudicada,26062,17963,44025
Cerrada,1,55,56
Desierta (o art. 3 ó 9 Ley 19.886),0,138,138
Revocada,0,7,7
All,26063,18163,44226


## 4. Pipeline modular de limpieza y transformación de datos
Se ejecuta la función `limpiar_datos_f2(df_raw)` de `src/proyecto.py`. Sus etapas y justificaciones son:
1. **Exclusión de columnas 100% vacías:** Se descartan `LicitacionBaseTipo`, `ContratoRenovable` y `UnidadTiempoRenovacion`.
2. **Neutralización de fechas anómalas:** Los 7.613 valores de `FechaEstimadaEvaluacionOfertas` fijados en el año 1900 corresponden a fechas anómalas para el periodo estudiado, cuyo significado oficial no se ha confirmado; se convierten a `NaT`.
3. **Casting a datetime:** Las columnas de fecha se convierten a formato de fecha/hora de pandas (`datetime64[ns]`), permitiendo cálculos de plazos.
4. **Normalización de texto:** Se remueven espacios en blanco residuales (`str.strip()`).
5. **Preservación de categorías:** Se preserva la etiqueta `NoClasificado` en `TamanoProveedor` (3.410 ofertas), evitando imputaciones artificiales.
6. **Ingeniería de variables:**
   - `oferta_ganadora`: booleano (`True` si `ResultadoOferta == 'Ganadora'`).
   - `licitacion_adjudicada`: booleano (`True` si `EstadoLicitacion == 'Adjudicada'`).
   - `plazo_cierre_dias`: cálculo de días entre publicación y cierre del proceso.
Los NA parciales se conservan sin imputar. Las fechas usadas se convierten explícitamente a datetime64[ns]; una fecha no interpretable detiene el proceso para revisión. Los demás campos temporales quedan fuera del cálculo actual. No se escala porque se calculan frecuencias y proporciones. No se eliminan filas por compartir NroLicitacion: un proceso reúne varios ítems y ofertas. La clave única de oferta/ítem sigue pendiente de contraste con el esquema del reporte.


In [6]:
df_procesado = limpiar_datos_f2(df_raw)
print("Dimensiones tras preprocesamiento:", df_procesado.shape)

cols_excluidas = ["LicitacionBaseTipo", "ContratoRenovable", "UnidadTiempoRenovacion"]
presentes = [c for c in cols_excluidas if c in df_procesado.columns]
assert len(presentes) == 0, f"Error: columnas vacías aún presentes: {presentes}"
print("OK: Columnas vacías excluidas correctamente.")

print("Tipo inferido FechaPublicacion:", df_procesado["FechaPublicacion"].dtype)
print("Tipo inferido FechaCierre:", df_procesado["FechaCierre"].dtype)
print("Tipo inferido FechaAdjudicacion:", df_procesado["FechaAdjudicacion"].dtype)

plazo_stats = df_procesado["plazo_cierre_dias"].describe()
print("\nEstadísticos de plazo_cierre_dias:")
display(plazo_stats.to_frame())
print("Fechas de evaluación faltantes tras limpieza:", int(df_procesado.FechaEstimadaEvaluacionOfertas.isna().sum()))
print("Los plazos anteriores están ponderados por registros de oferta, no por licitaciones únicas.")


Dimensiones tras preprocesamiento: (44226, 74)
OK: Columnas vacías excluidas correctamente.
Tipo inferido FechaPublicacion: datetime64[ns]
Tipo inferido FechaCierre: datetime64[ns]
Tipo inferido FechaAdjudicacion: datetime64[ns]

Estadísticos de plazo_cierre_dias:


,plazo_cierre_dias
count,44226.000000
mean,14.384128
std,6.862530
min,4.917241
25%,10.082613
50%,12.035603
75%,19.987572
max,61.032607


Fechas de evaluación faltantes tras limpieza: 44226
Los plazos anteriores están ponderados por registros de oferta, no por licitaciones únicas.


## 5. Análisis exploratorio de proporciones de éxito
Con los datos limpios, comparamos la tasa de adjudicación según el tamaño del proveedor en licitaciones que culminaron efectivamente en adjudicación (`licitacion_adjudicada == True`), evitando distorsiones por procesos desiertos o cancelados.

In [7]:
from src.proyecto import tabla_proporciones

tabla_resumen_tamano = tabla_proporciones(df_procesado, "TamanoProveedor")
print("Proporciones por tamaño en procesos adjudicados (denominador: resultados válidos):")
display(tabla_resumen_tamano)
# La misma función sirve para comparar otra variable, sin duplicar el cálculo.
display(tabla_proporciones(df_procesado, "TipoLicitacion"))

Proporciones por tamaño en procesos adjudicados (denominador: resultados válidos):


,Ganadora,Perdedora,Sin resultado válido,All,% Ganadora,% Perdedora
TamanoProveedor,,,,,,
Grande,12575,6255,0,18830,66.78,33.22
Mediana,4982,4001,0,8983,55.46,44.54
Micro,1680,1554,0,3234,51.95,48.05
NoClasificado,1803,1576,0,3379,53.36,46.64
Pequeña,5022,4577,0,9599,52.32,47.68
All,26062,17963,0,44025,59.20,40.80


,Ganadora,Perdedora,Sin resultado válido,All,% Ganadora,% Perdedora
TipoLicitacion,,,,,,
Licitación Privada Mayor a 1000 UTM,62,10,0,72,86.11,13.89
Licitación Privada Mayor a 5000 (I2),1,1,0,2,50.00,50.00
Licitación Privada entre 100 y 1000 UTM.,11,1,0,12,91.67,8.33
Licitación Pública Entre 100 y 1000 UTM (LE),14686,8904,0,23590,62.26,37.74
Licitación Pública Mayor 1000 UTM (LP),8066,6328,0,14394,56.04,43.96
Licitación Pública Mayor a 5000 (LR),1652,1559,0,3211,51.45,48.55
Licitación Pública Menor a 100 UTM (L1),1584,1160,0,2744,57.73,42.27
All,26062,17963,0,44025,59.20,40.80


## 6. Validación técnica y verificación del código
Para satisfacer el criterio de *Validación técnica y verificación del código*, se ejecutan pruebas automáticas que comprueban el comportamiento del pipeline en:
1. **Caso Normal:** Verificación integral de 6 reglas de calidad sobre el dataset real mediante `validar_dataset_procesado`.
2. **Caso Límite (Edge Case):** Comportamiento frente a fechas anómalas 1900 y registros con proveedor `NoClasificado`.
3. **Caso Excepción 1 (Columnas ausentes):** Intento de procesar un dataset sin columna obligatoria, verificando captura de `KeyError`.
4. **Caso Excepción 2 (Dataset vacío):** Intento de procesar un DataFrame sin registros, verificando captura de `ValueError`.

In [8]:
pruebas_f2 = []

# 1. Caso Normal
res_val = validar_dataset_procesado(df_procesado, len(df_raw))
pruebas_f2.append({
    "Caso": "Caso Normal (Dataset Real)",
    "Tipo": "Validación integral",
    "Resultado": res_val["estado"],
    "Detalle": f"{res_val['reglas_superadas']} reglas superadas sobre {res_val['filas_validadas']:,} registros"
})

# 2. Caso Límite
df_limite = pd.DataFrame({
    "NroLicitacion": ["LIC-TEST-01", "LIC-TEST-02"],
    "TipoLicitacion": ["LE", "LP"],
    "TamanoProveedor": ["NoClasificado", "Micro"],
    "ResultadoOferta": ["Ganadora", "Perdedora"],
    "EstadoLicitacion": ["Adjudicada", "Adjudicada"],
    "FechaPublicacion": ["2026-03-01 10:00:00.000", "2026-03-02 12:00:00.000"],
    "FechaCierre": ["2026-03-15 18:00:00.000", "2026-03-20 18:00:00.000"],
    "FechaEstimadaEvaluacionOfertas": ["1900-01-01 00:00:00.000", np.nan]
})
df_limite_proc = limpiar_datos_f2(df_limite)
assert df_limite_proc["FechaEstimadaEvaluacionOfertas"].isna().all()
assert df_limite_proc.loc[0, "TamanoProveedor"] == "NoClasificado"
pruebas_f2.append({
    "Caso": "Fechas de 1900 y categoría NoClasificado",
    "Tipo": "Tratamiento de fechas anómalas y conservación de categorías",
    "Resultado": "OK",
    "Detalle": "Fecha 1900 neutralizada a NaT y categoría NoClasificado preservada intacta"
})

# 3. Caso Excepción: Columna obligatoria ausente
df_sin_resultado = df_limite.drop(columns=["ResultadoOferta"])
try:
    limpiar_datos_f2(df_sin_resultado)
except KeyError as err:
    pruebas_f2.append({
        "Caso": "Caso Excepción: Columna faltante",
        "Tipo": "Manejo de errores de esquema",
        "Resultado": "OK",
        "Detalle": f"Captura controlada de KeyError ({err})"
    })
else:
    raise AssertionError("Fallo en prueba: no se detectó la columna ausente.")

# 4. Caso Excepción: Dataset sin filas
df_vacio = pd.DataFrame(columns=df_limite.columns)
try:
    limpiar_datos_f2(df_vacio)
except ValueError as err:
    pruebas_f2.append({
        "Caso": "Caso Excepción: DataFrame vacío",
        "Tipo": "Manejo de conjuntos vacíos",
        "Resultado": "OK",
        "Detalle": f"Captura controlada de ValueError ({err})"
    })
else:
    raise AssertionError("Fallo en prueba: no se detectó el DataFrame vacío.")

display(pd.DataFrame(pruebas_f2))
print("OK: Todas las pruebas de verificación del código fueron superadas exitosamente.")
# Regresiones de limpieza y validación: verificar resultados, no solo ejecución.
texto = df_limite.copy()
texto['TamanoProveedor'] = pd.Series([' Micro ', None], dtype='string')
texto['ContratoRenovable'] = ['SI', None]
antes = texto.copy(deep=True)
limpio = limpiar_datos_f2(texto)
assert limpio.loc[0, 'TamanoProveedor'] == 'Micro'
assert pd.isna(limpio.loc[1, 'TamanoProveedor'])
assert limpio.loc[0, 'ContratoRenovable'] == 'SI'
pd.testing.assert_frame_equal(texto, antes)
pruebas_f2.append({'Caso': 'Texto str, NA y columna con datos', 'Resultado': 'OK'})

invalida = df_limite.copy()
invalida.loc[0, 'FechaPublicacion'] = 'fecha inválida'
try:
    limpiar_datos_f2(invalida)
except ValueError:
    pruebas_f2.append({'Caso': 'Fecha no interpretable rechazada', 'Resultado': 'OK'})
else:
    raise AssertionError('No se detectó la fecha inválida')

base = limpiar_datos_f2(df_limite)
casos = {}
casos['Pérdida de filas'] = base.iloc[:1].copy()
casos['Indicador incorrecto'] = base.copy()
casos['Indicador incorrecto'].loc[0, 'oferta_ganadora'] = False
casos['Plazo incorrecto'] = base.copy()
casos['Plazo incorrecto'].loc[0, 'plazo_cierre_dias'] = 999
negativo = df_limite.copy()
negativo.loc[0, 'FechaCierre'] = '2026-02-01'
casos['Plazo negativo'] = limpiar_datos_f2(negativo)
casos['Derivada ausente'] = base.drop(columns=['plazo_cierre_dias'])
casos['Identificador nulo'] = base.copy()
casos['Identificador nulo'].loc[0, 'NroLicitacion'] = None
for nombre, entrada in casos.items():
    try:
        validar_dataset_procesado(entrada, len(df_limite))
    except AssertionError:
        pruebas_f2.append({'Caso': nombre, 'Resultado': 'OK'})
    else:
        raise AssertionError(f'No se detectó: {nombre}')
display(pd.DataFrame(pruebas_f2).fillna(''))
assert all(p['Resultado'] == 'OK' for p in pruebas_f2)


from src.proyecto import tabla_proporciones, normalizar_categorias, convertir_fechas
muestra_tasas = pd.DataFrame({"Grupo": ["A", "A", "B", "C"],
    "EstadoLicitacion": ["Adjudicada", "Adjudicada", "Adjudicada", "Revocada"],
    "ResultadoOferta": ["Ganadora", "Perdedora", None, "Ganadora"]})
tasas = tabla_proporciones(muestra_tasas, "Grupo")
assert tasas.loc["A", "% Ganadora"] == 50
assert tasas.loc["B", "All"] == 0 and pd.isna(tasas.loc["B", "% Ganadora"])
assert tasas.loc["B", "Sin resultado válido"] == 1
assert tasas.loc["All", "All"] == 2 and "C" not in tasas.index
assert pd.isna(tabla_proporciones(muestra_tasas.iloc[:0], "Grupo").loc["All", "% Ganadora"])
# Las etapas también admiten columnas distintas de las del proyecto.
aux = pd.DataFrame({"Etiqueta": [" A ", None], "Fecha": ["1900-01-01", "2026-03-01"]})
assert normalizar_categorias(aux, ["Etiqueta"]).loc[0, "Etiqueta"] == "A"
assert pd.isna(convertir_fechas(aux, ["Fecha"], {"Fecha": [1900]}).loc[0, "Fecha"])
print("OK: funciones reutilizadas con otras columnas y grupos sin resultados válidos.")


,Caso,Tipo,Resultado,Detalle
0,Caso Normal (Dataset Real),Validación integral,OK,"6 reglas superadas sobre 44,226 registros"
1,Fechas de 1900 y categoría NoClasificado,Tratamiento de fechas anómalas y conservación ...,OK,Fecha 1900 neutralizada a NaT y categoría NoCl...
2,Caso Excepción: Columna faltante,Manejo de errores de esquema,OK,"Captura controlada de KeyError (""Faltan column..."
3,Caso Excepción: DataFrame vacío,Manejo de conjuntos vacíos,OK,Captura controlada de ValueError (No es posibl...


OK: Todas las pruebas de verificación del código fueron superadas exitosamente.


,Caso,Tipo,Resultado,Detalle
0,Caso Normal (Dataset Real),Validación integral,OK,"6 reglas superadas sobre 44,226 registros"
1,Fechas de 1900 y categoría NoClasificado,Tratamiento de fechas anómalas y conservación ...,OK,Fecha 1900 neutralizada a NaT y categoría NoCl...
2,Caso Excepción: Columna faltante,Manejo de errores de esquema,OK,"Captura controlada de KeyError (""Faltan column..."
3,Caso Excepción: DataFrame vacío,Manejo de conjuntos vacíos,OK,Captura controlada de ValueError (No es posibl...
4,"Texto str, NA y columna con datos",,OK,
5,Fecha no interpretable rechazada,,OK,
6,Pérdida de filas,,OK,
7,Indicador incorrecto,,OK,
8,Plazo incorrecto,,OK,
9,Plazo negativo,,OK,


OK: funciones reutilizadas con otras columnas y grupos sin resultados válidos.


## 6.1. Conversión de categorías a indicadores numéricos

Aplicamos **one-hot encoding** a `TipoLicitacion` y `TamanoProveedor` mediante `pandas.get_dummies` (The pandas development team, s. f.). Cada categoría genera una columna entera: 1 indica pertenencia y 0 indica ausencia. No asignamos números consecutivos porque introducirían un orden o una distancia que no corresponde al tipo de licitación. Aunque los tamaños de empresa admiten un orden, aquí se representan por separado, incluida la categoría `NoClasificado`.

Conservamos las columnas originales y el dataset procesado usado para las proporciones. La versión codificada se exporta como un archivo adicional. Esta transformación muestra la representación numérica y permite explorar su uso posterior; no es necesaria para calcular las proporciones actuales. Si se entrena un modelo, habrá que ajustar el codificador solo con el conjunto de entrenamiento y definir el manejo de categorías nuevas. No se elimina una categoría de referencia en esta etapa; esa decisión dependerá del modelo.

Los faltantes deben resolverse explícitamente antes de codificar: no se interpretan como ausencia de todas las categorías. A continuación se muestran ejemplos antes/después y se comprueba que cada registro active exactamente una categoría por variable.


In [9]:
from src.proyecto import codificar_nominales, validar_codificacion

columnas_nominales = ("TipoLicitacion", "TamanoProveedor")
antes_codificacion = df_procesado.copy(deep=True)
df_codificado = codificar_nominales(df_procesado)
indicadores = [c for c in df_codificado if c not in df_procesado.columns]
print("Antes (categorías):")
ejemplos = df_procesado.drop_duplicates(list(columnas_nominales)).head(6).index
display(df_procesado.loc[ejemplos, list(columnas_nominales)])
print("Después (se conservan las categorías y se añaden indicadores):")
display(df_codificado.loc[ejemplos, list(columnas_nominales) + indicadores])

pd.testing.assert_frame_equal(df_procesado, antes_codificacion)
display(validar_codificacion(df_procesado, df_codificado, columnas_nominales))

# Ejemplo pequeño con índice repetido y NoClasificado: preservar filas y categorías.
mini = pd.DataFrame({"TipoLicitacion": ["LE", "LP"],
                     "TamanoProveedor": ["Micro", "NoClasificado"]}, index=[7, 7])
mini_cod = codificar_nominales(mini)
assert mini_cod["oh_TipoLicitacion__LE"].tolist() == [1, 0]
assert mini_cod["oh_TamanoProveedor__NoClasificado"].tolist() == [0, 1]
assert mini_cod.index.tolist() == [7, 7]
for caso, entrada, error in [
    ("Vacío", mini.iloc[:0], ValueError),
    ("Columna ausente", mini.drop(columns="TipoLicitacion"), KeyError),
    ("Faltante", mini.assign(TamanoProveedor=[None, "Micro"]), ValueError),
    ("Doble codificación", mini_cod, ValueError),
]:
    try:
        codificar_nominales(entrada)
    except error:
        print(f"OK: {caso}")
    else:
        raise AssertionError(f"No se detectó el caso: {caso}")
print(f"OK: {len(indicadores)} indicadores 0/1, filas y categorías originales conservadas.")
print("Dimensiones de la versión codificada:", df_codificado.shape)


# Verificar que la validación detecte indicadores corruptos.
mini_incorrecto = mini_cod.copy()
mini_incorrecto.iloc[0, mini_incorrecto.columns.get_loc("oh_TipoLicitacion__LE")] = 0
try:
    validar_codificacion(mini, mini_incorrecto, columnas_nominales)
except AssertionError:
    print("OK: indicador corrupto detectado")
else:
    raise AssertionError("No se detectó el indicador corrupto")


Antes (categorías):


,TipoLicitacion,TamanoProveedor
0,Licitación Pública Mayor a 5000 (LR),Grande
2,Licitación Pública Mayor a 5000 (LR),Pequeña
7,Licitación Pública Entre 100 y 1000 UTM (LE),Mediana
8,Licitación Pública Entre 100 y 1000 UTM (LE),Grande
21,Licitación Pública Entre 100 y 1000 UTM (LE),Pequeña
33,Licitación Pública Entre 100 y 1000 UTM (LE),Micro


Después (se conservan las categorías y se añaden indicadores):


,TipoLicitacion,TamanoProveedor,oh_TipoLicitacion__Licitación Privada Mayor a 1000 UTM,oh_TipoLicitacion__Licitación Privada Mayor a 5000 (I2),oh_TipoLicitacion__Licitación Privada entre 100 y 1000 UTM.,oh_TipoLicitacion__Licitación Pública Entre 100 y 1000 UTM (LE),oh_TipoLicitacion__Licitación Pública Mayor 1000 UTM (LP),oh_TipoLicitacion__Licitación Pública Mayor a 5000 (LR),oh_TipoLicitacion__Licitación Pública Menor a 100 UTM (L1),oh_TamanoProveedor__Grande,oh_TamanoProveedor__Mediana,oh_TamanoProveedor__Micro,oh_TamanoProveedor__NoClasificado,oh_TamanoProveedor__Pequeña
0,Licitación Pública Mayor a 5000 (LR),Grande,0,0,0,0,0,1,0,1,0,0,0,0
2,Licitación Pública Mayor a 5000 (LR),Pequeña,0,0,0,0,0,1,0,0,0,0,0,1
7,Licitación Pública Entre 100 y 1000 UTM (LE),Mediana,0,0,0,1,0,0,0,0,1,0,0,0
8,Licitación Pública Entre 100 y 1000 UTM (LE),Grande,0,0,0,1,0,0,0,1,0,0,0,0
21,Licitación Pública Entre 100 y 1000 UTM (LE),Pequeña,0,0,0,1,0,0,0,0,0,0,0,1
33,Licitación Pública Entre 100 y 1000 UTM (LE),Micro,0,0,0,1,0,0,0,0,0,1,0,0


{'estado': 'OK', 'indicadores': 12, 'filas': 44226}

OK: Vacío
OK: Columna ausente
OK: Faltante
OK: Doble codificación
OK: 12 indicadores 0/1, filas y categorías originales conservadas.
Dimensiones de la versión codificada: (44226, 86)
OK: indicador corrupto detectado


## 7. Exportación del dataset procesado y registro de trazabilidad
El conjunto procesado se almacena en `data/processed/licitaciones_salud_marzo_2026_procesado.csv` y se computa su huella SHA-256 para permitir trazabilidad del archivo en las siguientes entregas.

In [10]:
ruta_salida = raiz / "data" / "processed" / "licitaciones_salud_marzo_2026_procesado.csv"
info_exp = exportar_datos_procesados(df_procesado, ruta_salida)
display(pd.DataFrame([info_exp]))
print("Dataset exportado exitosamente a:", info_exp["ruta"])
print(f"Dimensiones exportadas: {info_exp['filas']:,} filas x {info_exp['columnas']} columnas ({info_exp['tamanio_mb']} MB)")
print("SHA-256 verificado:", info_exp["sha256"])

ruta_codificada = raiz / "data" / "processed" / "licitaciones_salud_marzo_2026_codificado.csv"
info_cod = exportar_datos_procesados(df_codificado, ruta_codificada)
display(pd.DataFrame([info_cod]))
# Comprobar que el CSV exportado conserva los indicadores enteros.
lectura_indicadores = pd.read_csv(ruta_codificada, sep=";", encoding="latin-1", usecols=indicadores)
pd.testing.assert_frame_equal(lectura_indicadores, df_codificado[indicadores].reset_index(drop=True), check_dtype=False)
assert all(pd.api.types.is_integer_dtype(t) for t in lectura_indicadores.dtypes)
print("OK: indicadores verificados tras exportar y volver a leer el CSV.")


,archivo,ruta,filas,columnas,tamanio_mb,sha256
0,licitaciones_salud_marzo_2026_procesado.csv,C:\Trabajos\sumativo-1\data\processed\licitaci...,44226,74,68.16,7e835d6725d8c4f9aa64ad97a294dc38fcc341e71373f5...


Dataset exportado exitosamente a: C:\Trabajos\sumativo-1\data\processed\licitaciones_salud_marzo_2026_procesado.csv
Dimensiones exportadas: 44,226 filas x 74 columnas (68.16 MB)
SHA-256 verificado: 7e835d6725d8c4f9aa64ad97a294dc38fcc341e71373f544c7f6b8ba2ff31dc5


,archivo,ruta,filas,columnas,tamanio_mb,sha256
0,licitaciones_salud_marzo_2026_codificado.csv,C:\Trabajos\sumativo-1\data\processed\licitaci...,44226,86,69.17,f430f9267861f4f9211b0cded233985485e4526f45d76a...


OK: indicadores verificados tras exportar y volver a leer el CSV.


## 8. Hallazgos técnicos y articulación con Fases 3 y 4
1. **Calidad de datos y trazabilidad:** Se validó la integridad de las 44.226 observaciones de ofertas del sector Salud. El tratamiento sistemático de nulos y fechas anómalas reduce errores de preparación, pero no elimina posibles sesgos del conjunto de datos.
2. **Patrones observados:** Las empresas catalogadas como **Grande** presentan una tasa de éxito de adjudicación (66.78%) mayor en los registros observados a las **Medianas** (55.46%), **Pequeñas** (52.32%) y **Micro** (51.95%).
3. **Próximos pasos (F3 y F4):** En la Fase 3 se desarrollarán visualizaciones de distribuciones y pruebas estadísticas para contrastar si estas diferencias son estadísticamente significativas por tipo de licitación.

## 9. Referencias bibliográficas (APA 7.ª edición)
- ChileCompra. (s. f.). *Datos abiertos: Descargas*. Recuperado el 12 de septiembre de 2026, de https://datos-abiertos.chilecompra.cl/descargas
- McKinney, W. (2022). *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3.ª ed.). O'Reilly Media.
- NumPy Developers. (s. f.). *NumPy documentation*. Recuperado el 12 de septiembre de 2026, de https://numpy.org/doc/stable/
- Samuel, S., & Mietchen, D. (2024). Computational reproducibility of Jupyter notebooks from biomedical publications. *GigaScience*, 13, giad113. https://doi.org/10.1093/gigascience/giad113
- The pandas development team. (s. f.). *pandas documentation*. Recuperado el 12 de septiembre de 2026, de https://pandas.pydata.org/docs/

## Organización de funciones reutilizables

`limpiar_datos_f2` coordina `excluir_columnas_vacias`, `convertir_fechas`, `normalizar_categorias` y `generar_variables_derivadas`. Las frecuencias y proporciones se calculan con `tablas_frecuencia` y `tabla_proporciones`; esta última se utiliza tanto por tamaño como por tipo de licitación. `codificar_nominales` y `validar_codificacion` separan transformación y comprobación. La lectura, el hash y la exportación también son funciones. Las celdas conservan la configuración, los ejemplos y la presentación de resultados.

## Pendiente antes de entregar
Faltan dos materiales docentes verificables y sus citas. Completar el recordatorio amarillo del informe; no se consideran cumplidas esas referencias.